In [1]:
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理函数，用于自回归语言模型
    
    参数:
        text: 输入文本
        n: 滑动窗口大小（特征序列长度）
    
    返回:
        vocab: 词汇表字典（词 -> ID）
        (features, labels): 特征列表和标签列表
    """
    # 1. 将文本转换为小写，去除标点符号（保留字母和空格）
    text = text.lower()
    text = ''.join([c for c in text if c.isalpha() or c.isspace()])
    
    # 2. 按空格分词
    tokens = text.split()
    tokens = [t for t in tokens if t]  # 去除空字符串
    
    # 3. 构建词汇表（按出现频率排序，分配整数ID，从0开始）
    word_counts = Counter(tokens)
    # 按词频降序排序，词频相同则按字母顺序排序
    sorted_words = sorted(word_counts.keys(), key=lambda x: (-word_counts[x], x))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 用滑动窗口生成长度为n的特征序列和对应的下一个词标签
    features = []
    labels = []
    
    for i in range(len(tokens) - n):
        feature = tokens[i:i+n]
        label = tokens[i+n]
        features.append(feature)
        labels.append(label)
    
    return vocab, (features, labels)

def test_preprocess_text():
    """测试文本预处理函数"""
    print("=" * 60)
    print("文本预处理函数测试")
    print("=" * 60)
    
    # 测试示例1：简单文本
    print("\n[1] 测试示例：\"The time machine\"")
    print("-" * 40)
    
    text1 = "The time machine"
    n1 = 2
    vocab1, (features1, labels1) = preprocess_text(text1, n1)
    
    print(f"输入文本: '{text1}'")
    print(f"滑动窗口n: {n1}")
    print(f"词汇表: {vocab1}")
    print(f"特征列表: {features1}")
    print(f"标签列表: {labels1}")
    
    # 测试示例2：带标点的文本
    print("\n[2] 测试带标点的文本")
    print("-" * 40)
    
    text2 = "Hello, World! This is a test. Test test."
    n2 = 3
    vocab2, (features2, labels2) = preprocess_text(text2, n2)
    
    print(f"输入文本: '{text2}'")
    print(f"词汇表: {vocab2}")
    print(f"特征列表 (前3个): {features2[:3]}")
    print(f"标签列表 (前3个): {labels2[:3]}")
    print(f"特征总数: {len(features2)}")
    
    # 测试示例3：多空格和空行
    print("\n[3] 测试多空格和空行")
    print("-" * 40)
    
    text3 = "I   love   deep    learning  "
    n3 = 2
    vocab3, (features3, labels3) = preprocess_text(text3, n3)
    
    print(f"输入文本: '{text3}'")
    print(f"词汇表: {vocab3}")
    print(f"特征列表: {features3}")
    print(f"标签列表: {labels3}")
    
    # 测试示例4：验证词频排序
    print("\n[4] 验证词频排序")
    print("-" * 40)
    
    text4 = "a a a b b c d e"
    vocab4, _ = preprocess_text(text4, 2)
    print(f"输入文本: '{text4}'")
    print(f"词汇表(按词频排序): {vocab4}")
    print("词频: a=3, b=2, c=1, d=1, e=1")
    
    # 测试示例5：n=1的情况
    print("\n[5] 测试n=1的情况")
    print("-" * 40)
    
    text5 = "The quick brown fox"
    n5 = 1
    vocab5, (features5, labels5) = preprocess_text(text5, n5)
    
    print(f"输入文本: '{text5}'")
    print(f"滑动窗口n: {n5}")
    print(f"特征列表: {features5}")
    print(f"标签列表: {labels5}")
    
    print("\n文本预处理函数测试完成！")

if __name__ == '__main__':
    test_preprocess_text()

文本预处理函数测试

[1] 测试示例："The time machine"
----------------------------------------
输入文本: 'The time machine'
滑动窗口n: 2
词汇表: {'machine': 0, 'the': 1, 'time': 2}
特征列表: [['the', 'time']]
标签列表: ['machine']

[2] 测试带标点的文本
----------------------------------------
输入文本: 'Hello, World! This is a test. Test test.'
词汇表: {'test': 0, 'a': 1, 'hello': 2, 'is': 3, 'this': 4, 'world': 5}
特征列表 (前3个): [['hello', 'world', 'this'], ['world', 'this', 'is'], ['this', 'is', 'a']]
标签列表 (前3个): ['is', 'a', 'test']
特征总数: 5

[3] 测试多空格和空行
----------------------------------------
输入文本: 'I   love   deep    learning  '
词汇表: {'deep': 0, 'i': 1, 'learning': 2, 'love': 3}
特征列表: [['i', 'love'], ['love', 'deep']]
标签列表: ['deep', 'learning']

[4] 验证词频排序
----------------------------------------
输入文本: 'a a a b b c d e'
词汇表(按词频排序): {'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4}
词频: a=3, b=2, c=1, d=1, e=1

[5] 测试n=1的情况
----------------------------------------
输入文本: 'The quick brown fox'
滑动窗口n: 1
特征列表: [['the'], ['quick'], ['brown']]
标签列表

In [2]:
import torch
import torch.nn.functional as F

def rnn_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    RNN单元前向传播
    
    参数:
        x_t: 当前时刻输入，形状 (batch_size, input_size)
        h_prev: 上一时刻隐藏状态，形状 (batch_size, hidden_size)
        W_hx: 输入到隐藏层的权重，形状 (hidden_size, input_size)
        W_hh: 隐藏层到隐藏层的权重，形状 (hidden_size, hidden_size)
        b_h: 隐藏层偏置，形状 (hidden_size,)
    
    返回:
        h_t: 当前时刻隐藏状态，形状 (batch_size, hidden_size)
        cache: 缓存中间结果用于反向传播
    """
    # 计算预激活
    # 注意：W_hx 形状是 (hidden_size, input_size)
    # x_t 形状是 (batch_size, input_size)
    # 所以需要 x_t @ W_hx.T 或直接用 F.linear
    pre_act = torch.mm(x_t, W_hx.t()) + torch.mm(h_prev, W_hh.t()) + b_h
    
    # tanh激活
    h_t = torch.tanh(pre_act)
    
    # 缓存中间结果
    cache = {
        'x_t': x_t,
        'h_prev': h_prev,
        'h_t': h_t,        # 缓存h_t而不是pre_act更好
        'pre_act': pre_act  # 也可以保留，但导数计算用h_t更稳定
    }
    
    return h_t, cache

def rnn_backward(dh_next, cache, W_hx, W_hh):
    """
    RNN单元单步反向传播（不更新权重）
    
    参数:
        dh_next: 上游梯度（损失对h_t的梯度），形状 (batch_size, hidden_size)
        cache: 前向传播缓存的中间结果
        W_hx: 输入到隐藏层的权重，形状 (hidden_size, input_size)
        W_hh: 隐藏层到隐藏层的权重，形状 (hidden_size, hidden_size)
    
    返回:
        dx_t: 输入梯度，形状 (batch_size, input_size)
        dh_prev: 上一隐藏状态梯度，形状 (batch_size, hidden_size)
        dW_hx: 输入权重梯度，形状 (hidden_size, input_size)
        dW_hh: 隐藏层权重梯度，形状 (hidden_size, hidden_size)
        db_h: 偏置梯度，形状 (hidden_size,)
    """
    x_t = cache['x_t']
    h_prev = cache['h_prev']
    h_t = cache['h_t']
    
    # tanh的导数: d(tanh(x))/dx = 1 - tanh(x)^2 = 1 - h_t^2
    dtanh = 1 - h_t ** 2
    
    # 预激活的梯度（链式法则）
    d_pre_act = dh_next * dtanh  # 形状: (batch_size, hidden_size)
    
    # 计算各梯度
    # dx_t: d_pre_act @ W_hx，形状 (batch_size, input_size)
    dx_t = torch.mm(d_pre_act, W_hx)
    
    # dh_prev: d_pre_act @ W_hh，形状 (batch_size, hidden_size)
    dh_prev = torch.mm(d_pre_act, W_hh)
    
    # dW_hx: d_pre_act.T @ x_t，形状 (hidden_size, input_size)
    dW_hx = torch.mm(d_pre_act.t(), x_t)
    
    # dW_hh: d_pre_act.T @ h_prev，形状 (hidden_size, hidden_size)
    dW_hh = torch.mm(d_pre_act.t(), h_prev)
    
    # db_h: sum over batch dimension，形状 (hidden_size,)
    db_h = d_pre_act.sum(dim=0)
    
    return dx_t, dh_prev, dW_hx, dW_hh, db_h

def test_rnn_unit():
    """测试RNN单元的前向传播和反向传播"""
    print("=" * 60)
    print("RNN单元前向传播与反向传播测试")
    print("=" * 60)
    
    # 参数设置
    batch_size = 2
    input_size = 3
    hidden_size = 4
    
    print("\n[1] 参数设置")
    print("-" * 40)
    print(f"批量大小: {batch_size}")
    print(f"输入维度: {input_size}")
    print(f"隐藏层维度: {hidden_size}")
    
    # 初始化随机数据
    torch.manual_seed(42)
    x_t = torch.randn(batch_size, input_size)
    h_prev = torch.randn(batch_size, hidden_size)
    W_hx = torch.randn(hidden_size, input_size)
    W_hh = torch.randn(hidden_size, hidden_size)
    b_h = torch.randn(hidden_size)
    
    print("\n[2] 前向传播测试")
    print("-" * 40)
    
    h_t, cache = rnn_forward(x_t, h_prev, W_hx, W_hh, b_h)
    print(f"输入x_t形状: {x_t.shape}")
    print(f"上一隐藏状态h_prev形状: {h_prev.shape}")
    print(f"W_hx形状: {W_hx.shape}")
    print(f"W_hh形状: {W_hh.shape}")
    print(f"b_h形状: {b_h.shape}")
    print(f"输出h_t形状: {h_t.shape}")
    
    print("\n[3] 反向传播测试")
    print("-" * 40)
    
    dh_next = torch.randn_like(h_t)
    print(f"上游梯度dh_next形状: {dh_next.shape}")
    
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_backward(dh_next, cache, W_hx, W_hh)
    
    print(f"\n计算得到的梯度形状:")
    print(f"  dx_t: {dx_t.shape}")
    print(f"  dh_prev: {dh_prev.shape}")
    print(f"  dW_hx: {dW_hx.shape}")
    print(f"  dW_hh: {dW_hh.shape}")
    print(f"  db_h: {db_h.shape}")
    
    print("\n[4] 梯度验证（与PyTorch自动求导对比）")
    print("-" * 40)
    
    # 使用PyTorch自动求导计算梯度
    x_t_autograd = x_t.clone().requires_grad_(True)
    h_prev_autograd = h_prev.clone().requires_grad_(True)
    W_hx_autograd = W_hx.clone().requires_grad_(True)
    W_hh_autograd = W_hh.clone().requires_grad_(True)
    b_h_autograd = b_h.clone().requires_grad_(True)
    
    # 前向传播
    pre_act_autograd = torch.mm(x_t_autograd, W_hx_autograd.t()) + \
                       torch.mm(h_prev_autograd, W_hh_autograd.t()) + b_h_autograd
    h_t_autograd = torch.tanh(pre_act_autograd)
    
    # 反向传播
    h_t_autograd.backward(dh_next)
    
    # 对比梯度
    print("梯度差异（应接近0）:")
    
    dx_diff = torch.norm(dx_t - x_t_autograd.grad).item()
    dh_diff = torch.norm(dh_prev - h_prev_autograd.grad).item()
    dW_hx_diff = torch.norm(dW_hx - W_hx_autograd.grad).item()
    dW_hh_diff = torch.norm(dW_hh - W_hh_autograd.grad).item()
    db_diff = torch.norm(db_h - b_h_autograd.grad).item()
    
    print(f"  ||dx_t - dx_t_autograd|| = {dx_diff:.8f}")
    print(f"  ||dh_prev - dh_prev_autograd|| = {dh_diff:.8f}")
    print(f"  ||dW_hx - dW_hx_autograd|| = {dW_hx_diff:.8f}")
    print(f"  ||dW_hh - dW_hh_autograd|| = {dW_hh_diff:.8f}")
    print(f"  ||db_h - db_h_autograd|| = {db_diff:.8f}")
    
    # 检查是否通过
    tolerance = 1e-6
    all_close = all([
        dx_diff < tolerance,
        dh_diff < tolerance,
        dW_hx_diff < tolerance,
        dW_hh_diff < tolerance,
        db_diff < tolerance
    ])
    
    if all_close:
        print(f"\n所有梯度计算正确！差异 < {tolerance}")
    else:
        print(f"\n梯度计算存在差异，超过容差 {tolerance}")
    
    print("\n[5] 公式说明")
    print("-" * 40)
    print("前向传播:")
    print("  a_t = x_t @ W_hx^T + h_prev @ W_hh^T + b_h")
    print("  h_t = tanh(a_t)")
    print()
    print("反向传播:")
    print("  dL/da_t = dL/dh_t ⊙ (1 - h_t²)")
    print("  dL/dx_t = dL/da_t @ W_hx")
    print("  dL/dh_prev = dL/da_t @ W_hh")
    print("  dL/dW_hx = (dL/da_t)^T @ x_t")
    print("  dL/dW_hh = (dL/da_t)^T @ h_prev")
    print("  dL/db_h = sum(dL/da_t, dim=0)")
    
    return all_close

if __name__ == '__main__':
    success = test_rnn_unit()
    print("\n" + "=" * 60)
    if success:
        print("RNN单元实现完全正确！")
    else:
        print("RNN单元实现存在问题，请检查！")
    print("=" * 60)

RNN单元前向传播与反向传播测试

[1] 参数设置
----------------------------------------
批量大小: 2
输入维度: 3
隐藏层维度: 4

[2] 前向传播测试
----------------------------------------
输入x_t形状: torch.Size([2, 3])
上一隐藏状态h_prev形状: torch.Size([2, 4])
W_hx形状: torch.Size([4, 3])
W_hh形状: torch.Size([4, 4])
b_h形状: torch.Size([4])
输出h_t形状: torch.Size([2, 4])

[3] 反向传播测试
----------------------------------------
上游梯度dh_next形状: torch.Size([2, 4])

计算得到的梯度形状:
  dx_t: torch.Size([2, 3])
  dh_prev: torch.Size([2, 4])
  dW_hx: torch.Size([4, 3])
  dW_hh: torch.Size([4, 4])
  db_h: torch.Size([4])

[4] 梯度验证（与PyTorch自动求导对比）
----------------------------------------
梯度差异（应接近0）:
  ||dx_t - dx_t_autograd|| = 0.00000000
  ||dh_prev - dh_prev_autograd|| = 0.00000000
  ||dW_hx - dW_hx_autograd|| = 0.00000000
  ||dW_hh - dW_hh_autograd|| = 0.00000000
  ||db_h - db_h_autograd|| = 0.00000000

所有梯度计算正确！差异 < 1e-06

[5] 公式说明
----------------------------------------
前向传播:
  a_t = x_t @ W_hx^T + h_prev @ W_hh^T + b_h
  h_t = tanh(a_t)

反向传播:
  dL/da_t = d

In [3]:
import torch
import torch.nn as nn

class BidirectionalRNNEncoder(nn.Module):
    """
    双向RNN编码器
    
    参数:
        input_dim: 输入特征维度
        hidden_dim: 单向隐藏层维度
        num_layers: RNN层数（默认1）
        bidirectional: 是否双向（默认True）
    """
    
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super(BidirectionalRNNEncoder, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 双向RNN层
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False  # 输入格式: (seq_len, batch, input_dim)
        )
    
    def forward(self, X):
        """
        双向RNN前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回:
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            seq_rep: 最终时间步的拼接隐藏状态（序列表示），形状 (batch, 2*hidden_dim)
        """
        # 前向传播
        # outputs: (seq_len, batch, 2*hidden_dim)
        # h_n: (2*num_layers, batch, hidden_dim)
        outputs, h_n = self.rnn(X)
        
        # 获取最终时间步的隐藏状态
        # 前向最后一个时间步: h_n[-2, :, :]
        # 后向最后一个时间步（即序列第一个位置）: h_n[-1, :, :]
        if self.num_layers > 1:
            # 如果多层，取最后一层的双向状态
            forward_final = h_n[2 * self.num_layers - 2, :, :]  # 前向最后一层
            backward_final = h_n[2 * self.num_layers - 1, :, :]  # 后向最后一层
        else:
            forward_final = h_n[0, :, :]  # 前向
            backward_final = h_n[1, :, :]  # 后向
        
        # 拼接前向和后向隐藏状态作为序列表示
        seq_rep = torch.cat([forward_final, backward_final], dim=1)
        
        return outputs, seq_rep

class BidirectionalRNNEncoderManual(nn.Module):
    """
    手动实现的双向RNN编码器
    
    参数:
        input_dim: 输入特征维度
        hidden_dim: 单向隐藏层维度
    """
    
    def __init__(self, input_dim, hidden_dim):
        super(BidirectionalRNNEncoderManual, self).__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        
        # 前向RNN参数
        self.W_hx_f = nn.Parameter(torch.randn(hidden_dim, input_dim))
        self.W_hh_f = nn.Parameter(torch.randn(hidden_dim, hidden_dim))
        self.b_h_f = nn.Parameter(torch.randn(hidden_dim))
        
        # 后向RNN参数
        self.W_hx_b = nn.Parameter(torch.randn(hidden_dim, input_dim))
        self.W_hh_b = nn.Parameter(torch.randn(hidden_dim, hidden_dim))
        self.b_h_b = nn.Parameter(torch.randn(hidden_dim))
        
        # 初始化参数
        self._init_weights()
    
    def _init_weights(self):
        """初始化权重"""
        nn.init.xavier_uniform_(self.W_hx_f)
        nn.init.xavier_uniform_(self.W_hh_f)
        nn.init.zeros_(self.b_h_f)
        
        nn.init.xavier_uniform_(self.W_hx_b)
        nn.init.xavier_uniform_(self.W_hh_b)
        nn.init.zeros_(self.b_h_b)
    
    def forward(self, X):
        """
        手动实现双向RNN前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, input_dim)
        
        返回:
            outputs: 每个时间步的拼接隐藏状态，形状 (seq_len, batch, 2*hidden_dim)
            seq_rep: 最终时间步的拼接隐藏状态（序列表示），形状 (batch, 2*hidden_dim)
        """
        seq_len, batch_size, _ = X.shape
        
        # 前向RNN
        h_f = torch.zeros(batch_size, self.hidden_dim, device=X.device)
        outputs_f = []
        
        for t in range(seq_len):
            x_t = X[t]
            h_f = torch.tanh(torch.mm(x_t, self.W_hx_f.t()) + 
                             torch.mm(h_f, self.W_hh_f.t()) + self.b_h_f)
            outputs_f.append(h_f)
        
        outputs_f = torch.stack(outputs_f, dim=0)  # (seq_len, batch, hidden_dim)
        
        # 后向RNN
        h_b = torch.zeros(batch_size, self.hidden_dim, device=X.device)
        outputs_b = []
        
        for t in range(seq_len - 1, -1, -1):
            x_t = X[t]
            h_b = torch.tanh(torch.mm(x_t, self.W_hx_b.t()) + 
                             torch.mm(h_b, self.W_hh_b.t()) + self.b_h_b)
            outputs_b.append(h_b)
        
        # 反转回原始顺序
        outputs_b = torch.stack(outputs_b[::-1], dim=0)  # (seq_len, batch, hidden_dim)
        
        # 拼接前向和后向输出
        outputs = torch.cat([outputs_f, outputs_b], dim=2)  # (seq_len, batch, 2*hidden_dim)
        
        # 序列表示：前向最后一个 + 后向第一个（即后向最后计算的）
        seq_rep = torch.cat([outputs_f[-1], outputs_b[0]], dim=1)
        
        return outputs, seq_rep

def test_bidirectional_rnn():
    """测试双向RNN编码器"""
    print("=" * 60)
    print("双向RNN编码器测试")
    print("=" * 60)
    
    # 参数设置
    seq_len = 5
    batch_size = 2
    input_dim = 3
    hidden_dim = 4
    
    print("\n[1] 参数设置")
    print("-" * 40)
    print("序列长度: %d" % seq_len)
    print("批量大小: %d" % batch_size)
    print("输入维度: %d" % input_dim)
    print("隐藏层维度: %d" % hidden_dim)
    
    # 生成随机输入
    torch.manual_seed(42)
    X = torch.randn(seq_len, batch_size, input_dim)
    print("\n输入形状: %s" % str(X.shape))
    
    print("\n[2] 使用PyTorch内置RNN测试")
    print("-" * 40)
    
    encoder = BidirectionalRNNEncoder(input_dim, hidden_dim)
    outputs, seq_rep = encoder(X)
    
    print("输出形状: %s" % str(outputs.shape))
    print("序列表示形状: %s" % str(seq_rep.shape))
    print("\n输出示例 (第一个样本前3个时间步):")
    print(outputs[:, 0, :])
    print("\n序列表示示例 (两个样本):")
    print(seq_rep)
    
    print("\n[3] 手动实现双向RNN测试")
    print("-" * 40)
    
    encoder_manual = BidirectionalRNNEncoderManual(input_dim, hidden_dim)
    outputs_manual, seq_rep_manual = encoder_manual(X)
    
    print("输出形状: %s" % str(outputs_manual.shape))
    print("序列表示形状: %s" % str(seq_rep_manual.shape))
    
    print("\n[4] 验证输出维度正确性")
    print("-" * 40)
    
    # 验证形状
    assert outputs.shape == (seq_len, batch_size, 2 * hidden_dim), "输出形状错误"
    assert seq_rep.shape == (batch_size, 2 * hidden_dim), "序列表示形状错误"
    assert outputs_manual.shape == (seq_len, batch_size, 2 * hidden_dim), "手动实现输出形状错误"
    assert seq_rep_manual.shape == (batch_size, 2 * hidden_dim), "手动实现序列表示形状错误"
    
    print("输出维度验证通过!")
    
    print("\n[5] 测试多层双向RNN")
    print("-" * 40)
    
    encoder_multi = BidirectionalRNNEncoder(input_dim, hidden_dim, num_layers=2)
    outputs_multi, seq_rep_multi = encoder_multi(X)
    
    print("层数: %d" % encoder_multi.num_layers)
    print("输出形状: %s" % str(outputs_multi.shape))
    print("序列表示形状: %s" % str(seq_rep_multi.shape))
    
    print("\n[6] 双向RNN结构说明")
    print("-" * 40)
    print("输入格式: (seq_len, batch_size, input_dim)")
    print("输出格式: (seq_len, batch_size, 2*hidden_dim)")
    print("序列表示: concat(前向最后时刻, 后向最后时刻)")
    print("形状: (batch_size, 2*hidden_dim)")
    
    print("\n双向RNN编码器测试完成！")

if __name__ == '__main__':
    test_bidirectional_rnn()

双向RNN编码器测试

[1] 参数设置
----------------------------------------
序列长度: 5
批量大小: 2
输入维度: 3
隐藏层维度: 4

输入形状: torch.Size([5, 2, 3])

[2] 使用PyTorch内置RNN测试
----------------------------------------
输出形状: torch.Size([5, 2, 8])
序列表示形状: torch.Size([2, 8])

输出示例 (第一个样本前3个时间步):
tensor([[ 0.5378, -0.9002, -0.5325,  0.4527,  0.6739, -0.1827,  0.4885, -0.2063],
        [-0.2958,  0.7955, -0.6811, -0.6325,  0.4656, -0.5337,  0.0865, -0.8847],
        [-0.8785,  0.6349, -0.6886, -0.4008, -0.3741,  0.5206,  0.1941, -0.9654],
        [-0.5378, -0.8103,  0.0140, -0.3549,  0.2826, -0.1199,  0.5454, -0.2329],
        [-0.6055,  0.2653,  0.3828, -0.4773, -0.6107,  0.6288,  0.5941, -0.6107]],
       grad_fn=<SliceBackward0>)

序列表示示例 (两个样本):
tensor([[-0.6055,  0.2653,  0.3828, -0.4773,  0.6739, -0.1827,  0.4885, -0.2063],
        [-0.2040,  0.2544, -0.4948, -0.6498, -0.7224,  0.5576,  0.7653, -0.6509]],
       grad_fn=<CatBackward0>)

[3] 手动实现双向RNN测试
----------------------------------------
输出形状: torch.Size([5, 2,

In [4]:
import torch
import torch.nn.functional as F

def cbow_forward(context_indices, target_indices, W, W_out):
    """
    CBOW模型前向传播和损失计算
    
    参数:
        context_indices: 上下文词索引列表，形状 (batch_size, context_size)
        target_indices: 中心词索引，形状 (batch_size,)
        W: 输入权重矩阵（嵌入矩阵），形状 (V, d)
        W_out: 输出权重矩阵，形状 (d, V)
    
    返回:
        loss: 交叉熵损失值
        logits: 输出得分，形状 (batch_size, V)
        hidden: 隐藏层（平均上下文向量），形状 (batch_size, d)
    """
    batch_size, context_size = context_indices.shape
    V, d = W.shape
    
    # 1. 获取上下文词的嵌入向量
    # context_indices: (batch_size, context_size) -> (batch_size, context_size, d)
    context_embeddings = W[context_indices]
    
    # 2. 计算平均上下文向量作为隐藏层
    # hidden: (batch_size, d)
    hidden = context_embeddings.mean(dim=1)
    
    # 3. 计算输出得分
    # logits: (batch_size, V)
    logits = torch.mm(hidden, W_out)
    
    # 4. 计算交叉熵损失
    loss = F.cross_entropy(logits, target_indices)
    
    return loss, logits, hidden

def test_cbow():
    """测试CBOW模型"""
    print("=" * 60)
    print("CBOW模型前向传播和损失计算测试")
    print("=" * 60)
    
    # 参数设置
    V = 10  # 词汇表大小
    d = 5   # 嵌入维度
    batch_size = 3
    context_size = 4  # 每个样本的上下文词数量
    
    print("\n[1] 参数设置")
    print("-" * 40)
    print("词汇表大小 V: %d" % V)
    print("嵌入维度 d: %d" % d)
    print("批量大小: %d" % batch_size)
    print("上下文窗口大小: %d" % context_size)
    
    # 初始化权重矩阵
    torch.manual_seed(42)
    W = torch.randn(V, d)  # 输入嵌入矩阵
    W_out = torch.randn(d, V)  # 输出权重矩阵
    
    print("\n[2] 权重矩阵形状")
    print("-" * 40)
    print("W (输入嵌入矩阵): %s" % str(W.shape))
    print("W_out (输出权重矩阵): %s" % str(W_out.shape))
    
    # 生成随机输入数据
    context_indices = torch.randint(0, V, (batch_size, context_size))
    target_indices = torch.randint(0, V, (batch_size,))
    
    print("\n[3] 输入数据")
    print("-" * 40)
    print("上下文词索引 (batch_size, context_size):")
    print(context_indices)
    print("\n中心词索引 (batch_size,):")
    print(target_indices)
    
    # 测试前向传播
    loss, logits, hidden = cbow_forward(context_indices, target_indices, W, W_out)
    
    print("\n[4] 前向传播结果")
    print("-" * 40)
    print("隐藏层形状: %s" % str(hidden.shape))
    print("隐藏层 (平均上下文向量):")
    print(hidden)
    
    print("\n输出得分形状: %s" % str(logits.shape))
    print("输出概率分布 (softmax后):")
    probs = F.softmax(logits, dim=1)
    print(probs)
    
    print("\n交叉熵损失: %.6f" % loss.item())
    
    print("\n[5] 验证计算过程")
    print("-" * 40)
    
    # 手动计算验证
    sample_idx = 0
    sample_context = context_indices[sample_idx]
    sample_target = target_indices[sample_idx]
    
    print("样本%d的上下文词索引: %s" % (sample_idx, sample_context.tolist()))
    print("样本%d的中心词索引: %d" % (sample_idx, sample_target.item()))
    
    # 手动计算平均上下文向量
    manual_hidden = W[sample_context].mean(dim=0)
    print("手动计算的隐藏向量: %s" % manual_hidden.tolist())
    print("前向传播的隐藏向量: %s" % hidden[sample_idx].tolist())
    
    # 验证是否一致
    hidden_diff = torch.norm(manual_hidden - hidden[sample_idx]).item()
    print("隐藏向量差异: %.6f" % hidden_diff)
    
    # 手动计算损失
    manual_logits = manual_hidden @ W_out
    manual_probs = F.softmax(manual_logits, dim=0)
    manual_loss = -torch.log(manual_probs[sample_target]).item()
    print("单个样本手动计算损失: %.6f" % manual_loss)
    
    print("\n[6] CBOW模型结构说明")
    print("-" * 40)
    print("输入: 上下文词索引 (batch_size, context_size)")
    print("步骤1: 获取嵌入 -> (batch_size, context_size, d)")
    print("步骤2: 平均池化 -> 隐藏层 (batch_size, d)")
    print("步骤3: 线性变换 -> 输出得分 (batch_size, V)")
    print("步骤4: softmax + 交叉熵 -> 损失")
    
    print("\n[7] 梯度测试")
    print("-" * 40)
    
    # 设置requires_grad=True进行梯度测试
    W_grad = W.clone().requires_grad_(True)
    W_out_grad = W_out.clone().requires_grad_(True)
    
    loss_grad, _, _ = cbow_forward(context_indices, target_indices, W_grad, W_out_grad)
    loss_grad.backward()
    
    print("W的梯度形状: %s" % str(W_grad.grad.shape))
    print("W_out的梯度形状: %s" % str(W_out_grad.grad.shape))
    print("W梯度范数: %.6f" % torch.norm(W_grad.grad).item())
    print("W_out梯度范数: %.6f" % torch.norm(W_out_grad.grad).item())
    
    print("\nCBOW模型测试完成！")

if __name__ == '__main__':
    test_cbow()

CBOW模型前向传播和损失计算测试

[1] 参数设置
----------------------------------------
词汇表大小 V: 10
嵌入维度 d: 5
批量大小: 3
上下文窗口大小: 4

[2] 权重矩阵形状
----------------------------------------
W (输入嵌入矩阵): torch.Size([10, 5])
W_out (输出权重矩阵): torch.Size([5, 10])

[3] 输入数据
----------------------------------------
上下文词索引 (batch_size, context_size):
tensor([[6, 0, 6, 8],
        [6, 8, 0, 6],
        [9, 0, 5, 9]])

中心词索引 (batch_size,):
tensor([5, 2, 0])

[4] 前向传播结果
----------------------------------------
隐藏层形状: torch.Size([3, 5])
隐藏层 (平均上下文向量):
tensor([[ 0.1467,  0.5712, -0.4830, -0.7931,  0.1842],
        [ 0.1467,  0.5712, -0.4830, -0.7931,  0.1842],
        [ 1.7281, -0.0678,  1.2506, -1.1855,  0.5340]])

输出得分形状: torch.Size([3, 10])
输出概率分布 (softmax后):
tensor([[1.2214e-01, 1.9646e-02, 1.0781e-02, 7.2587e-02, 3.5623e-01, 8.6844e-02,
         1.0457e-01, 6.2393e-02, 2.2737e-02, 1.4207e-01],
        [1.2214e-01, 1.9646e-02, 1.0781e-02, 7.2587e-02, 3.5623e-01, 8.6844e-02,
         1.0457e-01, 6.2393e-02, 2.2737e-02, 1.4

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    """
    多头注意力机制实现
    
    参数:
        d_model: 模型维度
        num_heads: 头的数量
    """
    
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        
        assert d_model % num_heads == 0, "d_model必须能被num_heads整除"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # 每个头的维度
        
        # 线性投影层
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        
        # 最终线性层
        self.W_O = nn.Linear(d_model, d_model)
    
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """
        缩放点积注意力
        
        参数:
            Q: 查询矩阵，形状 (batch, seq_len, d_k)
            K: 键矩阵，形状 (batch, seq_len, d_k)
            V: 值矩阵，形状 (batch, seq_len, d_k)
            mask: 掩码矩阵（可选）
        
        返回:
            output: 注意力输出，形状 (batch, seq_len, d_k)
            attn_weights: 注意力权重，形状 (batch, seq_len, seq_len)
        """
        # 计算注意力得分: (batch, seq_len, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32))
        
        # 应用掩码
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # 计算注意力权重
        attn_weights = F.softmax(scores, dim=-1)
        
        # 计算输出
        output = torch.matmul(attn_weights, V)
        
        return output, attn_weights
    
    def forward(self, X, mask=None):
        """
        多头注意力前向传播
        
        参数:
            X: 输入序列，形状 (seq_len, batch, d_model)
            mask: 掩码矩阵（可选）
        
        返回:
            output: 输出序列，形状 (seq_len, batch, d_model)
            attn_weights: 注意力权重
        """
        seq_len, batch_size, _ = X.shape
        
        # 1. 线性投影得到Q, K, V
        Q = self.W_Q(X)  # (seq_len, batch, d_model)
        K = self.W_K(X)
        V = self.W_V(X)
        
        # 2. 转换为batch_first格式并分割成多个头
        # (seq_len, batch, d_model) -> (batch, seq_len, num_heads, d_k)
        Q = Q.transpose(0, 1).view(batch_size, seq_len, self.num_heads, self.d_k)
        K = K.transpose(0, 1).view(batch_size, seq_len, self.num_heads, self.d_k)
        V = V.transpose(0, 1).view(batch_size, seq_len, self.num_heads, self.d_k)
        
        # 3. 对每个头计算注意力
        heads_output = []
        heads_attn = []
        
        for i in range(self.num_heads):
            q_i = Q[:, :, i, :]  # (batch, seq_len, d_k)
            k_i = K[:, :, i, :]
            v_i = V[:, :, i, :]
            
            output_i, attn_i = self.scaled_dot_product_attention(q_i, k_i, v_i, mask)
            heads_output.append(output_i)
            heads_attn.append(attn_i)
        
        # 4. 拼接所有头的输出: (batch, seq_len, d_model)
        output = torch.cat(heads_output, dim=-1)
        
        # 5. 最终线性层
        output = self.W_O(output)
        
        # 转换回 (seq_len, batch, d_model)
        output = output.transpose(0, 1)
        
        # 注意力权重
        attn_weights = torch.stack(heads_attn, dim=1)
        
        return output, attn_weights

def test_multi_head_attention():
    """测试多头注意力"""
    print("=" * 60)
    print("多头注意力 (Multi-Head Attention) 测试")
    print("=" * 60)
    
    # 参数设置
    d_model = 4
    num_heads = 2
    seq_len = 3
    batch_size = 2
    
    print("\n[1] 参数设置")
    print("-" * 40)
    print("模型维度 d_model: %d" % d_model)
    print("头的数量 num_heads: %d" % num_heads)
    print("每个头的维度 d_k: %d" % (d_model // num_heads))
    print("序列长度 seq_len: %d" % seq_len)
    print("批量大小 batch_size: %d" % batch_size)
    
    # 创建多头注意力实例
    mha = MultiHeadAttention(d_model, num_heads)
    print("\n多头注意力结构:")
    print(mha)
    
    # 生成随机输入
    torch.manual_seed(42)
    X = torch.randn(seq_len, batch_size, d_model)
    print("\n输入形状: %s" % str(X.shape))
    print("输入 X:")
    print(X)
    
    # 前向传播
    output, attn_weights = mha(X)
    
    print("\n[2] 前向传播结果")
    print("-" * 40)
    print("输出形状: %s" % str(output.shape))
    print("输出:")
    print(output)
    
    print("\n[3] 注意力权重")
    print("-" * 40)
    print("注意力权重形状: %s" % str(attn_weights.shape))
    print("头0的注意力权重 (样本0):")
    print(attn_weights[0, 0])
    print("\n头1的注意力权重 (样本0):")
    print(attn_weights[0, 1])
    
    print("\n[4] 验证输出维度")
    print("-" * 40)
    assert output.shape == X.shape, "输出形状与输入不匹配"
    print("输出形状验证通过！")
    
    print("\n[5] 多头注意力结构说明")
    print("-" * 40)
    print("输入: (seq_len, batch, d_model)")
    print("步骤1: 线性投影 -> Q, K, V")
    print("步骤2: 分割成头 -> (batch, seq_len, num_heads, d_k)")
    print("步骤3: 每个头计算缩放点积注意力")
    print("步骤4: 拼接 -> (batch, seq_len, d_model)")
    print("步骤5: 最终线性层 -> (seq_len, batch, d_model)")
    
    print("\n[6] 缩放点积注意力公式")
    print("-" * 40)
    print("Attention(Q,K,V) = softmax(Q @ K^T / sqrt(d_k)) @ V")
    
    print("\n多头注意力测试完成！")

if __name__ == '__main__':
    test_multi_head_attention()

多头注意力 (Multi-Head Attention) 测试

[1] 参数设置
----------------------------------------
模型维度 d_model: 4
头的数量 num_heads: 2
每个头的维度 d_k: 2
序列长度 seq_len: 3
批量大小 batch_size: 2

多头注意力结构:
MultiHeadAttention(
  (W_Q): Linear(in_features=4, out_features=4, bias=True)
  (W_K): Linear(in_features=4, out_features=4, bias=True)
  (W_V): Linear(in_features=4, out_features=4, bias=True)
  (W_O): Linear(in_features=4, out_features=4, bias=True)
)

输入形状: torch.Size([3, 2, 4])
输入 X:
tensor([[[ 1.9269,  1.4873,  0.9007, -2.1055],
         [ 0.6784, -1.2345, -0.0431, -1.6047]],

        [[ 0.3559, -0.6866, -0.4934,  0.2415],
         [-1.1109,  0.0915, -2.3169, -0.2168]],

        [[-0.3097, -0.3957,  0.8034, -0.6216],
         [-0.5920, -0.0631, -0.8286,  0.3309]]])

[2] 前向传播结果
----------------------------------------
输出形状: torch.Size([3, 2, 4])
输出:
tensor([[[-0.6094, -0.1858,  0.3175,  0.3578],
         [-0.2568, -0.0936,  0.4186,  0.4177]],

        [[-0.6361, -0.2620,  0.3214,  0.3045],
         [-0.2352, 